In [1]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 111.5 MB/s eta 0:00:00


In [4]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Page Configuration
st.set_page_config(
    page_title="Sales Forecasting Dashboard",
    page_icon="📈",
    layout="wide"
)

st.title("📈 Sales & Demand Forecasting Dashboard")
st.markdown("Machine Learning Project using Superstore Dataset")

# Upload Dataset
uploaded_file = st.file_uploader(
    "Upload Superstore CSV File",
    type=["csv"]
)

if uploaded_file is not None:

    # Load Dataset
    df = pd.read_csv(uploaded_file, encoding="latin1")

    st.success("Dataset Loaded Successfully")

    # Dataset Preview
    st.subheader("Dataset Preview")
    st.dataframe(df.head())

    # Convert Dates
    df['Order Date'] = pd.to_datetime(
        df['Order Date'],
        errors='coerce'
    )

    # KPI Section
    st.subheader("Business Overview")

    total_sales = df['Sales'].sum()
    total_profit = df['Profit'].sum()
    total_orders = len(df)

    col1, col2, col3 = st.columns(3)

    col1.metric(
        "Total Sales",
        f"${total_sales:,.2f}"
    )

    col2.metric(
        "Total Profit",
        f"${total_profit:,.2f}"
    )

    col3.metric(
        "Total Orders",
        total_orders
    )

    # Monthly Sales Trend
    st.subheader("Monthly Sales Trend")

    df['Month'] = df['Order Date'].dt.month

    monthly_sales = (
        df.groupby('Month')['Sales']
        .sum()
        .reset_index()
    )

    fig1, ax1 = plt.subplots(figsize=(8,4))

    ax1.plot(
        monthly_sales['Month'],
        monthly_sales['Sales'],
        marker='o'
    )

    ax1.set_title("Monthly Sales Trend")
    ax1.set_xlabel("Month")
    ax1.set_ylabel("Sales")

    st.pyplot(fig1)

    # Region Sales
    st.subheader("Region Wise Sales")

    region_sales = (
        df.groupby('Region')['Sales']
        .sum()
        .reset_index()
    )

    fig2, ax2 = plt.subplots(figsize=(8,4))

    sns.barplot(
        data=region_sales,
        x='Region',
        y='Sales',
        ax=ax2
    )

    ax2.set_title("Region Wise Sales")

    st.pyplot(fig2)

    # Category Sales
    st.subheader("Category Wise Sales")

    category_sales = (
        df.groupby('Category')['Sales']
        .sum()
        .reset_index()
    )

    fig3, ax3 = plt.subplots(figsize=(8,4))

    sns.barplot(
        data=category_sales,
        x='Category',
        y='Sales',
        ax=ax3
    )

    ax3.set_title("Category Wise Sales")

    st.pyplot(fig3)

    # Forecasting Section
    st.subheader("Sales Forecasting Model")

    daily_sales = (
        df.groupby('Order Date')['Sales']
        .sum()
        .reset_index()
    )

    daily_sales['Year'] = daily_sales['Order Date'].dt.year
    daily_sales['Month'] = daily_sales['Order Date'].dt.month
    daily_sales['Day'] = daily_sales['Order Date'].dt.day

    X = daily_sales[['Year', 'Month', 'Day']]
    y = daily_sales['Sales']

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    st.subheader("Model Performance")

    c1, c2 = st.columns(2)

    c1.metric("MAE", round(mae, 2))
    c2.metric("RMSE", round(rmse, 2))

    # Forecast Future Months
    st.subheader("Future Sales Forecast")

    months = st.slider(
        "Select Number of Months",
        1,
        12,
        6
    )

    future = pd.DataFrame({
        'Year': [2026] * months,
        'Month': list(range(1, months + 1)),
        'Day': [1] * months
    })

    forecast = model.predict(future)

    future['Forecast Sales'] = forecast

    st.dataframe(future)

    fig4, ax4 = plt.subplots(figsize=(8,4))

    ax4.plot(
        future['Month'],
        future['Forecast Sales'],
        marker='o'
    )

    ax4.set_title("Future Sales Forecast")
    ax4.set_xlabel("Month")
    ax4.set_ylabel("Forecast Sales")

    st.pyplot(fig4)

    st.success("Forecast Generated Successfully")

Overwriting app.py


In [5]:
!ls

app.py	sample_data


In [6]:
!streamlit run app.py &>/content/log.txt &

In [8]:

from pyngrok import ngrok

ngrok.set_auth_token("3ERdlh3ySszAFOzisN2rdoWIbEK_3Rx6HviHjGYH8EpNSRqmH")


In [9]:
public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://hunting-remedial-plausible.ngrok-free.dev" -> "http://localhost:8501"
